# <b> 02일차 인공지능을 활용한 의료데이터 분석 기초

In [ ]:
# Shared lecture path bootstrap
from pathlib import Path
from collections import deque
import os
import sys
import unicodedata

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

SEARCH_ROOTS = (
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path.home(),
)
PROJECT_NAMES = {"의료_이미지_2026", "의료_이미지 2026", "앨리스 - 의료이미지 (26.04)"}

def _norm(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def _is_project_root(path: Path) -> bool:
    return (path / "00_shared" / "utils" / "lecture_paths.py").exists()

def _find_project_root() -> Path:
    override = os.environ.get("PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if _is_project_root(root):
            return root
        raise FileNotFoundError("환경변수 PROJECT_ROOT가 올바른 강의 루트를 가리키지 않습니다.")

    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if _is_project_root(root):
            return root

    target_names = {_norm(name) for name in PROJECT_NAMES}
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        queue = deque([(base.resolve(), 0)])
        seen = set()
        while queue:
            current, depth = queue.popleft()
            if current in seen:
                continue
            seen.add(current)

            if _norm(current.name) in target_names and _is_project_root(current):
                return current
            if depth >= 4:
                continue

            try:
                children = [
                    child for child in current.iterdir()
                    if child.is_dir() and not child.name.startswith(".") and child.name not in {"__pycache__", ".ipynb_checkpoints"}
                ]
            except OSError:
                continue

            children.sort(key=lambda child: (_norm(child.name) not in target_names, child.name))
            queue.extend((child.resolve(), depth + 1) for child in children)

    raise FileNotFoundError(
        "PROJECT_ROOT를 찾지 못했습니다. Colab에서는 os.environ['PROJECT_ROOT']를 먼저 지정하세요."
    )

def _resolve_day_dir(name: str) -> Path:
    for candidate in {name, unicodedata.normalize("NFC", name), unicodedata.normalize("NFD", name)}:
        path = PROJECT_ROOT / candidate
        if path.exists():
            return path
    return PROJECT_ROOT / name

PROJECT_ROOT = _find_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
UTILS_ROOT = PROJECT_ROOT / "00_shared" / "utils"
if str(UTILS_ROOT) not in sys.path:
    sys.path.append(str(UTILS_ROOT))

from lecture_paths import DATA_ROOT, MODEL_ROOT, ensure_dir

DAY_DIR = _resolve_day_dir('02일차_의료데이터_탐색_기초실습')
DAY_OUTPUT_ROOT = ensure_dir(DAY_DIR / "99_실행산출물")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_ROOT: {MODEL_ROOT}")
print(f"DAY_OUTPUT_ROOT: {DAY_OUTPUT_ROOT}")


## 실행 환경 가이드

- 로컬 실행: 바로 위 bootstrap 셀부터 실행하고, 출력된 `PROJECT_ROOT`, `DATA_ROOT`, `DAY_OUTPUT_ROOT`를 먼저 확인합니다.
- Colab 실행: 가장 확실한 방법은 아래처럼 `PROJECT_ROOT`를 먼저 지정한 뒤 bootstrap 셀을 실행하는 것입니다.
- 자동 탐색은 현재 작업 폴더 상위와 `Colab Notebooks`, `MyDrive`, `Shareddrives` 아래의 `의료_이미지_2026` 폴더를 대상으로 합니다. 경로가 다르면 `PROJECT_ROOT`를 직접 지정하세요.

```python
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["PROJECT_ROOT"] = "/content/drive/MyDrive/강의자료/의료_이미지_2026"
```

- 실습 산출물은 기본적으로 해당 일차의 `99_실행산출물/`에 저장합니다.


## AI 도구 활용 체크포인트

이 구간은 현재 사용 가능한 AI 도구 아무거나 사용해도 됩니다. 특정 도구 이름보다 `질문 -> 검증 -> 수정` 흐름에 집중하세요.

### 질문 예시
- 이 데이터셋에서 먼저 확인해야 할 변수 구조와 기본 통계량을 표로 정리해줘.
- 지금 만든 시각화가 무엇을 보여주는지 비전공자도 이해할 수 있게 설명해줘.

### 직접 바꿔볼 값
- 표본으로 확인할 이미지 수 또는 행 수를 바꿔본다.
- 히스토그램, countplot, boxplot 중 한 가지 시각화 방식을 바꿔본다.

### 결과 해석 질문
- EDA 결과를 보고 어떤 변수나 이미지 패턴이 이후 예측 문제에서 중요할지 설명할 수 있는가?

### 현업 적용 질문
- 실제 의료 현장에서 비슷한 데이터를 받는다면 어떤 항목을 추가 수집하거나 먼저 확인할지 정리해본다.


## 사전 라이브러리 브리지

- `01_Pandas로_심혈관CSV_읽기.ipynb`: DataFrame, 결측치, 기본 통계 확인
- `02_Matplotlib로_심혈관CSV_그리기.ipynb`: 막대그래프, 히스토그램, 박스플롯 기초
- `03_Seaborn으로_심혈관패턴_보기.ipynb`: countplot, boxplot, heatmap 기초
- `pandas`, `matplotlib`, `seaborn`이 익숙하지 않다면 위 3개를 먼저 보고 메인 실습으로 넘어가세요.



##### 02일차 (1) 의료 이미지 수집 및 시각화 #####

## 사용 데이터셋 1: 뇌종양 MRI 데이터셋
- Kaggle 공개 데이터셋 (Brain Tumor Classification MRI)
- 링크: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
- 이미지 크기: 총 93.08MB
- 이미지 형식: JPG
- 클래스(라벨): <br> 1) Glioma (교종) <br> 2) Meningioma (수막종) <br> 3) Pituitary (뇌하수체 종양) <br> 4) no-tumor (종양없음)
- 활용기법: 데이터 수집, 전처리, 시각화

-----
    
##### 02일차 (2) 심장병 유무 예측 (선형회귀) #####

## 사용 데이터셋 2: 심장병 데이터셋
- Kaggle 공개 데이터셋 (Heart Disease Data for Health Research)
- 링크: https://www.kaggle.com/datasets/oktayrdeki/heart-disease
- 이미지 크기: 총 1.53MB
- 이미지 형식: CSV
- 클래스(라벨): 심장병 유무(예/아니오)
- 활용기법: 데이터 수집, 전처리, EDA(데이터 탐색), 리니어 리그레션
    
## 학습 목표: 의료 이미지 데이터를 수집하고 분석 및 시각화

----

# [1] 의료 이미지 수집 및 시각화

## 1. 라이브러리 불러오기 및 폴더경로 설정

#### <b> 이미지를 불러오고 시각화하기 위한 기본 라이브러리 불러오기

In [ ]:
import os                                # 파일/폴더 경로 조작, 디렉토리 탐색, 경로 합치기(os.path.join) 등
import random                            # 무작위 샘플링/난수 생성 (random.sample, random.choice 등)
import numpy as np                       # 수치 계산·배열 연산·선형대수 (np.array, np.random 등)
import pandas as pd                      # 표형 데이터 처리(DataFrame), CSV 입출력, 전처리
import matplotlib.pyplot as plt          # 시각화 라이브러리: 그래프/이미지 출력
from PIL import Image                    # 이미지 파일 열기/변환/저장
from IPython.display import display      # 주피터에서 DataFrame/이미지/HTML을 깔끔하게 표시


#### <b> 데이터셋 경로 및 클래스 정의
데이터셋의 경로를 상단에 지정해놓으면, 추후에 데이터의 위치가 바뀌었을때 손쉽게 수정이 가능합니다.
또한, 데이터위치의 상단폴더와 하단폴더를 별도 지정하면, 데이터의 저장위치가 바뀌었을 시, 상단 폴더만 수정하면 모든 폴더의 위치를 일일히 수정하지 않아도 된다는 장점이 있습니다.

In [ ]:
### 뇌종양 데이터셋 경로 정의
base_dir = str(DATA_ROOT / "brain_tumor_mri" / "Brain Tumor MRI" / "Training")
classes = ["glioma_tumor", "meningioma_tumor", "pituitary_tumor", "no_tumor"]

## 2. 시각화 함수 정의

#### <b> 샘플 이미지 시각화 함수 정의

여러 클래스에서 <b>대표 샘플 이미지를 동시에 그리드로 배치</b>해 보여줌으로써, 데이터셋이 <b>정상적으로 준비되었는지</b>와 클래스별 이미지의 <b>전반적 특징</b>을 빠르게 확인합니다. 일반적으로 각 클래스에서 일정 개수의 이미지를 <b>무작위로 추출</b>하여 2×N 또는 3×N 형태의 서브플롯으로 배치하고, 각 패널에는 <b>클래스 이름</b>을 제목으로 표시합니다.  


In [ ]:
### 시각화 함수
def show_samples(base_dir, classes, n_per_class=2, title="Dataset"):
    total = len(classes) * n_per_class        # 전체 출력할 이미지 수
    plt.figure(figsize=(4*3, 2*3))            # 출력 크기 지정 (가로, 세로)
    i = 1                                     # subplot 인덱스

    for cls in classes:                       # 클래스별로 반복
        class_dir = os.path.join(base_dir, cls)   # 클래스 폴더 경로
        imgs = sorted(os.listdir(class_dir))[:n_per_class]  # 해당 클래스의 앞쪽 이미지 일부 선택
        for img_name in imgs:                 # 선택된 이미지들 반복
            img_path = os.path.join(class_dir, img_name)    # 이미지 경로
            img = Image.open(img_path).convert("RGB")       # 이미지 읽기 (RGB 변환)
            plt.subplot(2, 4, i)               # 2행 4열 배치 중 i번째 위치
            plt.imshow(img, cmap="gray")       # 이미지 표시
            plt.title(cls)                     # 클래스 이름을 제목으로 표시
            plt.axis("off")                    # x, y 축 제거
            i += 1                             # 다음 subplot 위치로 이동

    plt.suptitle(title, fontsize=16)           # 전체 제목
    plt.tight_layout()                         # 레이아웃 자동 조정
    plt.show()                                 # 시각화 출력


## 3. 이미지 시각화

#### <b> 앞에서 만든 `show_samples` 함수를 이용해 Brain Tumor MRI 데이터셋에서 클래스별 샘플 이미지를 확인

앞서 정의한 <b>`show_samples` 함수</b>를 실행하면 Brain Tumor MRI 데이터셋의 <b>클래스별 샘플 이미지</b>가 <b>2행 × 4열(최대 8개)</b> 배치로 출력됩니다. 각 패널의 제목에는 <b>해당 클래스 이름</b>이 표시되어 있어, 데이터셋이 <b>정상적으로 로드되었는지</b>와 클래스별 이미지가 <b>어떤 형태적 특징</b>을 보이는지 한눈에 점검할 수 있습니다. 실행 결과는 아래 시각화와 같이 나타나며, 잘못된 라벨링이나 파일 손상 등 <b>데이터 품질 이슈</b>를 조기에 발견하는 데에도 유용합니다.


In [ ]:
### 실행 코드
show_samples(base_dir, classes, n_per_class=2, title="Brain Tumor MRI")

# [2] 심장병 데이터 탐색 및 심장병 유무 예측

## 1. 라이브러리 불러오기 및 폴더경로 설정

#### <b> 이미지를 불러오고 시각화하기 위한 기본 라이브러리 불러오기

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#### <b> 데이터셋 경로 및 클래스 정의
- 데이터셋의 경로를 상단에 지정해놓으면, 추후에 데이터의 위치가 바뀌었을때 손쉽게 수정이 가능합니다.
- 또한, 데이터위치의 상단폴더와 하단폴더를 별도 지정하면, 데이터의 저장위치가 바뀌었을 시, 상단 폴더만 수정하면 모든 폴더의 위치를 일일히 수정하지 않아도 된다는 장점이 있습니다.

In [ ]:
# 프로젝트 최상단 경로
base_dir = str(DATA_ROOT / "heart_disease")    #현재 데이터셋이 코드와 동일한 위치에 있어서 별도의 상단경로 지정하지 않음

# 데이터 파일 경로
path = str(DATA_ROOT / "heart_disease" / "heart_disease.csv")

## 2. 데이터 불러오기

#### <b> Pandas라이브러리를 통해 데이터 읽어오기

데이터 파일이 <b>CSV 형식</b>으로 제공될 경우, `pandas` 라이브러리의 `read_csv()` 함수를 사용하여 데이터를 불러옵니다. 이때 단순히 공백으로 표시된 값(`""`)은 <b>결측치(NaN)</b>로 자동 변환되도록 지정합니다. 반면 `'None'`이나 그 외 문자열은 그대로 유지하여, 불필요하게 결측치로 처리되지 않도록 합니다.  

이렇게 설정하면 실제로 비어 있는 값과 문자열 `"None"`을 구분하여 다룰 수 있어, 이후 전처리 과정에서 데이터 손상을 방지할 수 있습니다.  


In [ ]:
df =pd.read_csv(path, keep_default_na=False, na_values=[""])

#### <b> 데이터 기초 정보 확인

`info()` 메서드는 엑셀에서 요약 정보를 확인하는 것과 비슷한 역할을 합니다. 이 함수를 실행하면 각 컬럼이 <b>숫자형인지/문자형인지</b> 데이터 타입을 보여주고, <b>결측치 존재 여부</b>, <b>행과 열의 개수</b> 등 기본 구조를 한눈에 파악할 수 있습니다.  

머신러닝이나 인공지능 분석을 시작하기 전, 반드시 실행해보는 <b>기본 탐색 함수</b>로, 데이터 전처리 방향을 결정하는 데 중요한 기준을 제공합니다.  


In [ ]:
print(df.info())   

#### 위 정보 해석하기
- 행(Row) 개수: 총 10,000개의 데이터가 있음 (RangeIndex: 0 to 9999).
- 열(Column) 개수: 총 21개의 변수(피처 + 타깃)가 존재.
- 자료형(Dtype) 분포: <br>
float64 → 9개 (연속형 수치 변수, 예: Age, BMI 등) <br>
object → 12개 (범주형/문자 변수, 예: Gender, Smoking 등) <br>
- 결측치(Non-Null Count): <br>
대부분 변수에 약 20~30개 정도 결측치가 있음 (10,000 중 ~9970 수준). <br>
(예) Age: 9971 non-null → 29개 결측<br>
Cholesterol Level: 9970 non-null → 30개 결측<br>
Heart Disease Status: 10,000 non-null → 결측 없음 (타깃 변수는 완전함).<br>
- 메모리 사용량: 약 1.6 MB → 중간 규모 데이터로 메모리 부담 적음.

---

#### <b> 컬럼별 결측치 개수 확인

각 컬럼에는 결측치가 존재할 수 있으므로, <b>컬럼 단위로 결측치 개수를 집계</b>해 확인합니다. 이를 통해 어떤 변수에 결측치가 집중되어 있는지, 전처리 과정에서 <b>삭제</b> 또는 <b>보간(Imputation)</b>이 필요한지를 판단할 수 있습니다.  

결측치 분포를 먼저 확인하면, 단순 삭제가 가능한지 아니면 보간 기법을 적용해야 하는지를 결정할 수 있습니다.


In [ ]:
print("컬럼별 결측치 개수") # 결측치 개수 확인
print(df.isnull().sum())

#### <b> 데이터셋의 수치형(float) 변수들의 요약 통계량을 확인

데이터셋 내 <b>수치형(float) 변수</b>들에 대해 기본적인 <b>요약 통계량(descriptive statistics)</b>을 확인합니다. 이를 통해 각 변수의 <b>최솟값, 최댓값, 평균, 표준편차</b> 등을 한눈에 파악할 수 있으며, 값이 대체로 어떤 범위를 가지는지, 얼마나 퍼져 있는지를 이해할 수 있습니다.  

이러한 요약 통계는 <b>이상치(outlier) 탐지</b>나 <b>정규화/스케일링 여부 결정</b> 등 이후 데이터 전처리 과정의 중요한 기준이 됩니다.  


In [ ]:
# 수치형(float) 변수 요약 통계
num_summary = df.select_dtypes(include=['float64']).describe().T  # float 타입만 선택 후 통계 요약
num_summary = num_summary[['mean', 'min', 'max', 'std']]         # 주요 통계 지표만 선택
print(num_summary)                                               # 결과 출력

#### <b> 범주형(object) 변수들의 고유값 분포를 확인

데이터셋에서 <b>범주형(object) 변수</b>를 대상으로 각 컬럼이 어떤 <b>고유값(unique value)</b>들을 가지는지와 그 <b>분포(빈도수)</b>를 확인합니다. 이를 통해 각 범주가 데이터에 얼마나 포함되어 있는지 알 수 있으며, 특정 범주가 과도하게 많거나 적을 경우 <b>클래스 불균형</b> 문제를 미리 파악할 수 있습니다.  


In [ ]:
# 범주형(object) 변수 추출
cat_cols = df.select_dtypes(include=['object']).columns  

# 각 범주형 변수의 값 분포 출력
for col in cat_cols:
    print("---"*9)                             # 구분선 출력
    print(df[col].value_counts(dropna=False))  # NaN 포함 값 분포 출력


#### <b> 심장병 여부(Heart Disease Status)에 따라 수치형 변수들의 평균값을 비교

이 데이터는 환자의 다양한 <b>건강 관련 레코드</b>와 <b>심장병 유무</b>를 기록해놓은 데이터셋입니다. 따라서 여러 수치형 지표들을 종합적으로 활용하여 <b>심장병 발생 여부를 예측</b>할 수 있으며, 이를 위해 먼저 심장병 여부에 따른 변수 차이를 확인하는 과정이 필요합니다.  

심장병 환자 그룹과 건강한 그룹을 나누어 각 그룹의 <b>수치형 변수 평균값</b>을 비교합니다. 예를 들어 평균 나이, 평균 혈압, 평균 콜레스테롤 수치 등이 두 집단 간에 어떻게 다른지를 한눈에 확인할 수 있습니다.  

이러한 비교는 특정 건강 지표가 심장병 발생과 어떤 관련성을 가질 수 있는지 탐색하는 데 유용하며, 이후 통계 검정이나 머신러닝 모델링 과정에서 <b>잠재적 주요 피처</b>로 고려할 수 있는 근거가 됩니다.  


In [ ]:
# 'Heart Disease Status'별로 그룹을 나누고 수치형 변수의 평균값 계산
group_means = df.groupby("Heart Disease Status").mean(numeric_only=True)
print(group_means)


#### <b> 범주형 변수와 심장병 여부(Heart Disease Status) 간의 비율 관계를 확인

이 데이터셋은 환자의 다양한 <b>범주형 특성</b>(예: 성별, 흡연 여부, 운동 습관, 가족력 등)과 심장병 유무를 함께 기록하고 있습니다. 따라서 범주형 변수별로 심장병 환자와 비환자 간의 <b>비율 차이</b>를 확인하는 것은 매우 중요합니다.  

범주형 변수와 타깃 변수(심장병 여부)를 교차 분석하면, 특정 생활 습관이나 조건이 심장병 발생에 어떤 영향을 미치는지 <b>직관적으로 비교</b>할 수 있습니다.  

이 과정은 심장병 예측 모델을 만들 때, <b>범주형 특성의 중요도</b>를 이해하고 전처리 방향을 설정하는 데 도움이 됩니다.  


In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.drop("Heart Disease Status")

for col in cat_cols:
    print("----"*10)
    # 범주형 변수와 Heart Disease Status의 교차표 (비율 기반)
    print(pd.crosstab(df[col], df["Heart Disease Status"], normalize='columns'))


## 3. 데이터 시각화


이제 데이터를 다양한 관점에서 시각화해 봅니다. 이것은 단순히 숫자와 표를 보는 것보다 훨씬 직관적으로 <b>패턴과 특징</b>을 이해할 수 있게 합니다.  

시각화를 통해 변수 간의 <b>관계</b>, <b>이상치(Outlier)</b> 존재 여부, <b>클래스 분포</b> 등을 쉽게 파악할 수 있으며, 이는 데이터 전처리와 모델링 전략을 세우는 데 중요한 단서를 제공합니다.  

---

#### <b> 타깃 변수(심장병 유무)의 분포를 보여주는 막대그래프

데이터셋에서 <b>심장병이 있는 환자</b>와 <b>심장병이 없는 환자</b>의 수를 막대그래프로 시각화합니다. 이를 통해 두 집단의 <b>대략적인 비율</b>을 한눈에 확인할 수 있으며, 타깃 데이터가 <b>균형(balance)</b>을 이루고 있는지 혹은 <b>불균형(imbalance)</b> 문제가 있는지도 직관적으로 파악할 수 있습니다.  

시각화 결과 심장병이 없는 사람이 있는 사람보다 4배 정도 많은것을 볼 수 있습니다.  


In [ ]:
plt.figure(figsize=(5,4))  # 그래프 크기 지정
sns.countplot(x=df["Heart Disease Status"], palette="pastel")  # 클래스별 빈도 막대그래프
plt.title("Distribution of Heart Disease Status")  # 제목
plt.xlabel("Heart Disease Status")  # x축 이름
plt.ylabel("Count")  # y축 이름
plt.show()  # 출력


#### <b> 수치형 변수들의 히스토그램(histogram)

데이터셋에 포함된 <b>모든 수치형 변수</b>를 대상으로 히스토그램을 시각화합니다. 코드에서 `df[num_cols].hist()`를 사용했기 때문에, DataFrame 내의 모든 수치형 컬럼(float64, int64)이 자동으로 선택되어 각각 하나의 서브플롯(subplot)으로 출력됩니다.  

이를 통해 각 변수 값들의 <b>분포 형태</b>(정규분포, 치우침, 이상치 등)를 한눈에 확인할 수 있습니다. 이러한 분포 확인은 이후 <b>스케일링 필요성</b>이나 <b>데이터 변환(로그 변환 등)</b> 여부를 판단하는 데 중요한 기초 자료가 됩니다.  


In [ ]:
num_cols = df.select_dtypes(include=["float64", "int64"]).columns
df[num_cols].hist(figsize=(12,10), bins=20)
plt.suptitle("Histograms of Numeric Features", fontsize=16)
plt.show()

#### <b> 심장병 유무(Heart Disease Status)에 따른 콜레스테롤 수치 분포 비교

`boxplot`을 활용하여 <b>심장병 여부</b>에 따른 <b>콜레스테롤 수치 분포</b>를 비교합니다. x축에는 심장병 유무(0:No/1:Yes), y축에는 콜레스테롤 수치가 표시되며, 각 그룹별로 <b>중앙값, 사분위 범위, 이상치</b>를 시각적으로 확인할 수 있습니다.  

시각화 결과, 심장병이 있는 그룹과 없는 그룹 간의 콜레스테롤 분포는 <b>큰 차이가 없으며 거의 유사한 형태</b>를 보입니다. 이는 콜레스테롤 수치 단독으로는 심장병 여부를 구분하는 데 뚜렷한 기준이 되지 않을 수 있음을 시사합니다.  


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Heart Disease Status", y="Cholesterol Level", data=df, palette="Set2")
plt.title("Cholesterol by Heart Disease Status")
plt.show()

#### <b> 범주형 변수(성별)에 따라 심장질환 유무가 어떻게 분포하는지 시각화

<b>성별(Gender)</b>을 기준으로 환자들을 나누어, 심장병이 있는 그룹과 없는 그룹의 <b>분포 차이</b>를 시각화합니다. 카운트플롯을 사용하면 각 성별 내에서 심장병 유무 비율이 어떻게 다른지 직관적으로 확인할 수 있습니다.  

이를 통해 남성과 여성 집단 간에 심장질환 발생 양상이 유사한지, 혹은 한쪽 성별에서 상대적으로 더 높은 비율을 보이는지를 파악할 수 있습니다.  
    
시각화 결과, 남여모두 심장병이 없는 경우와 있는 경우의 비율이 4:1가량으로 거의 동일했으며, 시각화 부분 맨 처음에 확인한 전체 심장병 비율과도 거의 동일한 것으로 확인되었습니다. (실제로 이 정도로 비슷한 비율의 데이터는 raw 데이터라기 보다는 정제된 데이터에 가까울 것입니다.)


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="Gender", hue="Heart Disease Status", data=df, palette="muted")
plt.title("Gender vs Heart Disease Status")
plt.show()

## 3. 데이터 전처리



인공지능/머신러닝 모델을 학습하기 전에는 반드시 데이터를 <b>학습 가능한 형태</b>로 가공해야 합니다. 앞서 수행한 데이터 탐색과 시각화를 통해 데이터의 특징과 문제점을 확인했다면, 이제 그 결과를 반영하여 본격적인 전처리 단계로 들어갑니다.  

전처리 과정은 매우 다양하지만, 이 강의에서는 우선적으로 다음 두 가지를 다루겠습니다.  
1. <b>결측치 보간(Imputation)</b> — 데이터에 존재하는 빈값을 적절히 채워 넣어 학습 시 오류를 방지합니다.  
2. <b>데이터 분할(Splitting)</b> — 전체 데이터를 학습용과 평가용으로 나누어, 모델 성능을 공정하게 검증할 수 있도록 합니다.  

이 두 단계는 이후 모델링 단계로 넘어가기 전 반드시 거쳐야 하는 <b>기초 전처리 절차</b>입니다.  

---


### 3.1 <b> 결측치가 포함된 행 탐색

데이터셋에서 <b>결측치가 하나 이상 존재하는 행</b>을 찾아 그 개수를 출력합니다. 이를 통해 전체 데이터 중 결측치가 포함된 레코드가 얼마나 되는지 확인할 수 있습니다.  

또한, 결측치가 포함된 행들에 대해 <b>'Heart Disease Status'</b> 값의 분포를 함께 출력하여, 심장병 환자 그룹과 비환자 그룹 중 어느 쪽에서 결측치가 더 많이 발생했는지 확인합니다.  

이 과정은 결측치 처리 전략을 세울 때, 단순히 데이터 손실량을 고려하는 것뿐 아니라 타깃 분포에 어떤 영향을 줄 수 있는지도 함께 평가할 수 있게 해줍니다.  


In [ ]:
# 결측치가 하나 이상인 행 찾기
missing_rows = df[df.isnull().any(axis=1)]                      
print("결측치가 있는 행 개수:", len(missing_rows))

# 결측치가 있는 행 중에서 'Heart Disease Status' 값 분포 출력
print("\nHeart Disease Status 분포 (결측치 포함 행)")
print(missing_rows["Heart Disease Status"].value_counts())


### 3.2 <b> 결측치 핸들링

앞선 데이터 탐색에서 확인한 것처럼, 이 데이터셋에는 일부 <b>결측치</b>가 존재합니다. 결측치를 적절히 처리하지 않으면 모델 학습 과정에서 오류가 발생하거나 성능이 저하될 수 있기 때문에 반드시 보간 과정이 필요합니다.  

이번 단계에서는 두 가지 방법을 사용하여 결측치를 채워 넣습니다.  
1. <b>KNN Imputation</b> — 비슷한 샘플들의 값을 활용하여 결측치를 보간하는 방법  
2. <b>Median/Mode Imputation</b> — 수치형 변수는 중앙값(Median), 범주형 변수는 최빈값(Mode)으로 채우는 방법  

이 두 가지 접근법을 통해 각각의 장단점을 비교해보고, 어떤 방식이 데이터 특성에 더 적합한지 평가할 수 있습니다.  


#### <b> (1) 결측치(빈 값)를 KNN 알고리즘으로 채우는 함수

결측치가 존재하는 데이터를 처리하기 위해 <b>KNN(K-Nearest Neighbors) 알고리즘</b>을 활용합니다. 이 방법은 "비슷한 샘플은 비슷한 값을 가진다"라는 가정에 기반을 두고 있습니다.  

먼저 결측치가 포함된 행을 찾은 후, 유클리드 거리(Euclidean distance) 등을 이용해 <b>가장 가까운 K개의 이웃 데이터</b>를 탐색합니다.  
- 결측치가 <b>수치형 변수</b>일 경우, 이웃 값들의 <b>평균</b>으로 채웁니다.  
- 결측치가 <b>범주형 변수</b>(예: 원-핫 인코딩된 경우)일 경우, 이웃 값들의 <b>다수결</b>로 채웁니다.  

이 방식은 단순한 중앙값/최빈값 대체보다 데이터의 맥락을 더 잘 반영할 수 있으며, 데이터 손실을 최소화하면서 자연스러운 보간을 제공합니다.  


In [ ]:
### 결측치 처리 함수 : KNN Imputation

def preprocess_knn(df, target="Heart Disease Status", n_neighbors=5):
    # 타깃 변수는 제외하고 입력 피처만 추출, 범주형 변수는 원-핫 인코딩
    x = pd.get_dummies(df.drop(target, axis=1), drop_first=True)

    # 'No' → 0, 'Yes' → 1 로 변환
    y = df[target].map({"No":0, "Yes":1})

    # KNN 기반 결측치 대체기 생성 (기본: 주변 5개 이웃 평균)
    imputer = KNNImputer(n_neighbors=n_neighbors)

    # 결측치 대체 수행 후 DataFrame 형태로 복원
    x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

    return x_imputed, y  # 처리된 입력변수 x, 라벨 y 리턴


#### <b> (2) 결측치를 중앙값(Median) 또는 최빈값(Mode)으로 채우는 전처리 함수

결측치를 간단하면서도 안정적으로 처리하기 위해, 변수의 유형에 따라 서로 다른 방식으로 보간합니다.  

- <b>수치형 변수</b>는 해당 컬럼의 <b>중앙값(Median)</b>으로 채웁니다. 중앙값은 평균보다 극단값(Outlier)의 영향을 덜 받기 때문에 데이터 분포를 더 안정적으로 반영할 수 있습니다.  
- <b>범주형 변수</b>는 가장 많이 등장한 값, 즉 <b>최빈값(Mode)</b>으로 채웁니다.  

또한, 결측치가 원래 존재했는지를 기록하기 위해 <b>`_missing`</b>이라는 새로운 컬럼을 추가하여 0(없음)/1(있음)으로 표시합니다.  

이 방식은 알고리즘이 단순하고 계산이 빠르며, 데이터의 전체적인 분포를 크게 왜곡하지 않는다는 장점이 있어 기초 전처리에서 많이 활용됩니다.  


In [ ]:
def preprocess_median_mode(df, target="Heart Disease Status"):
    X = df.drop(target, axis=1).copy() # 타켓 열 제거
    y = df[target].map({"No":0, "Yes":1}) # 라벨을 숫자로 변환

    # 수치형 컬럼 처리
    for col in X.select_dtypes(include=['float64']): 
        X[col + "_missing"] = X[col].isnull().astype(int) # 결측치 여부(1/0) 기록
        X[col] = X[col].fillna(X[col].median()) # 중앙값으로 결측치 채우기

    # 범주형 컬럼 처리
    for col in X.select_dtypes(include=['object']):
        X[col + "_missing"] = X[col].isnull().astype(int) # 결측치 여부 기록
        X[col] = X[col].fillna(X[col].mode()[0]) # 최빈값으로 결측치 채우기

    # 범주형 변수를 원-핫 인코딩
    X = pd.get_dummies(X, drop_first=True) 
    return X, y


#### <b> 결측치 전처리 실행
위에서 정의한 2가지 보간법을 이용하여 각가가 전처리를 수행합니다.

In [ ]:
# KNN 방식
X_knn, y_knn = preprocess_knn(df)

# Median Mode 방식
X_mm, y_mm = preprocess_median_mode(df)


### 3.3 데이터 분할

머신러닝 모델을 올바르게 학습하고 평가하기 위해 데이터셋을 <b>학습용(train)</b>과 <b>평가용(test)</b>으로 나누는 단계입니다. 일반적으로 <b>전체 데이터의 80%</b>는 모델 학습에 사용하고, 나머지 <b>20%</b>는 모델 성능을 검증하는 데 사용합니다.  

이때 중요한 점은, 타깃(label)의 <b>클래스 분포</b>가 원래 데이터와 동일하게 유지되도록 <b>층화 분할(stratified split)</b>을 수행하는 것입니다. 이렇게 해야 모델이 학습 시 특정 클래스에 치우치지 않고, 실제 데이터 분포를 반영한 성능 평가가 가능합니다.  

데이터 분할은 모델의 <b>일반화 성능</b>을 공정하게 평가하기 위한 필수 과정입니다.  


In [ ]:
### KNN 결측치 처리 버전
X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(
    X_knn,                # 입력 데이터 (KNN으로 결측치 처리된 버전)
    y_knn,                # 라벨 (심장병 여부: 0/1)
    test_size=0.2,        # 전체 데이터 중 20%는 테스트용, 80%는 학습용
    random_state=42,      # 랜덤 시드 고정 (재현성 확보 → 다시 실행해도 같은 분할 결과 나옴)
    stratify=y_knn        # y 비율(Yes/No 분포)을 train/test에 동일하게 유지
)

### Median/Mode 결측치 처리 버전

X_train_mm, X_test_mm, y_train_mm, y_test_mm = train_test_split(
    X_mm,                 # 입력 데이터 (중앙값/최빈값으로 결측치 처리된 버전)
    y_mm,                 # 라벨 (심장병 여부: 0/1)
    test_size=0.2,        # 전체 데이터 중 20%는 테스트용, 80%는 학습용
    random_state=42,      # 랜덤 시드 고정
    stratify=y_mm         # 라벨 분포를 학습/테스트 세트 모두 동일하게 유지
)

## 4. 데이터 학습

#### <b> 선형 회귀(Linear Regression) 모델 정의 및 학습

앞서 결측치 보간을 두 가지 방식(KNN Imputation, Median/Mode Imputation)으로 처리한 데이터를 준비하였습니다. 따라서 이 두 가지 전처리 방식을 각각 적용한 데이터셋을 사용하여 <b>가장 기본적인 선형 회귀(Linear Regression) 모델</b>을 학습합니다.  

선형 회귀는 입력 변수(X)와 타깃 변수(y) 사이의 관계를 <b>직선 방정식( y = wX + b )</b>으로 근사하는 방법입니다. 즉, 주어진 데이터를 가장 잘 설명할 수 있는 직선을 찾아내는 것이 목표입니다.  

- **목적**: 입력 변수들의 선형 결합을 통해 타깃 값을 예측  
- **장점**: 수학적으로 간단하고 직관적이며, 학습 속도가 빠름  
- **단점**: <b>비선형 관계</b>나 복잡한 데이터를 잘 설명하지 못함
- **활용**: 연속형 값을 예측하는 문제(예: 혈압, 콜레스테롤 수치, 체질량지수 등)에서 기초 모델로 자주 사용 
    
이번 단계에서는 동일한 알고리즘을 사용하되 <b>데이터 전처리 방식</b>에 따라 모델 학습 결과가 어떻게 달라지는지를 비교할 수 있습니다. 이를 통해 결측치 처리 방법이 모델의 성능과 해석에 어떤 영향을 미치는지 파악할 수 있습니다.  

또한, 심장병 데이터셋의 수치형 피처들을 활용해 타깃 변수와의 관계를 선형적으로 학습시키고, 결과를 해석하는 과정을 진행합니다.  


In [ ]:
# 1) KNN 방식으로 결측치를 처리한 데이터를 학습시킬 모델
model_knn_linear = LinearRegression()

# 2) 중앙값(Median)/최빈값(Mode) 방식으로 결측치를 처리한 데이터를 학습시킬 모델
model_mm_linear = LinearRegression()


In [ ]:
# 1) KNN 기반 결측치 처리 데이터로 학습
model_knn_linear.fit(X_train_knn, y_train_knn)

# 2) Median/Mode 기반 결측치 처리 데이터로 학습
model_mm_linear.fit(X_train_mm, y_train_mm)


## 5. 모델을 활용한 심장병 예측 및 평가

#### <b> 테스트 데이터 예측

앞서 두 가지 방식(KNN Imputation, Median/Mode Imputation)으로 결측치를 처리한 데이터셋을 기반으로 각각 선형 회귀 모델을 학습시켰습니다. 이제 이 두 모델에 <b>테스트 데이터</b>를 넣어 예측을 수행합니다.  

이를 통해 동일한 알고리즘(Linear Regression)이라도 <b>결측치 처리 방법</b>에 따라 예측 결과가 어떻게 달라지는지를 비교할 수 있습니다. 이는 전처리 방식이 모델 성능과 결과 해석에 미치는 영향을 이해하는 중요한 단계입니다.  


In [ ]:
# 1) KNN 기반 결측치 처리 데이터로 학습한 모델을 사용해 테스트 데이터 예측
y_pred_knn_linear = model_knn_linear.predict(X_test_knn)

# 2) Median/Mode 기반 결측치 처리 데이터로 학습한 모델을 사용해 테스트 데이터 예측
y_pred_mm_linear  = model_mm_linear.predict(X_test_mm)


#### <b> 테스트 데이터 성능 평가
- 총 3가지 지표를 통해 성능을 평가합니다.
1) MSE (Mean Squared Error, 평균제곱오차): 
    - 실제값 𝑦_𝑖와 예측값 𝑦^_𝑖 의 차이를 제곱해서 평균을 계산합니다.
    - 값이 작을수록 모델 예측이 실제에 가깝다는 뜻입니다.
    - 제곱이 들어가므로 큰 오차에 더 민감합니다.
2) MAE (Mean Absolute Error, 평균절대오차) :
    - 오차의 절댓값 평균을 말합니다.
    - 해석이 쉽고, MSE보다 이상치(Outlier)에 덜 민감합니다.
    - 평균적으로 얼마나 틀렸나를 직관적으로 보여주는 지표입니다.
3) R²(결정계수, Coefficient of Determination) : 
    - 1에 가까울수록 좋은 모델
    - 데이터의 분산(총 변동성) 중에서 모델이 설명할 수 있는 비율
    - 예: 𝑅^2=0.85 ->데이터 변동성의 85%를 모델이 설명한다는 의미
    - 0 이면 아무 설명 못 하고, 음수가 나오면 평균값으로 예측하는 것보다 못한 모델입니다.

In [ ]:
print("KNN-LinearRegression 회귀지표:",
      "MSE=", mean_squared_error(y_test_knn, y_pred_knn_linear),
      "MAE=", mean_absolute_error(y_test_knn, y_pred_knn_linear),
      "R2=",  r2_score(y_test_knn, y_pred_knn_linear))

print("MM-LinearRegression 회귀지표:",
      "MSE=", mean_squared_error(y_test_mm, y_pred_mm_linear),
      "MAE=", mean_absolute_error(y_test_mm, y_pred_mm_linear),
      "R2=",  r2_score(y_test_mm, y_pred_mm_linear))

#### <b> 예측 결과 해석

- 두 모델 모두 <b>MSE(평균제곱오차)</b>와 <b>MAE(평균절대오차)</b> 값이 거의 동일하게 나타났습니다. 즉, 결측치 처리 방식의 차이가 성능에 미치는 영향은 미미합니다.  
- <b>R² 값이 0보다 작다</b>는 것은, 이 선형 회귀 모델이 단순히 평균값으로만 예측하는 모델보다도 성능이 낮다는 의미입니다. 즉, 현재 데이터와 선형 회귀 모델 간에는 뚜렷한 선형적 관계가 잘 드러나지 않고 있습니다.  
- 결론적으로, 이 데이터셋에서는 단순 선형 회귀 모델이 적합하지 않을 수 있으며, <b>비선형 모델(Decision Tree, Random Forest 등)</b>이나 더 정교한 방법을 시도하는 것이 필요합니다.  
    
----


#### <b> 분류 지표 계산 (Accuracy, Precision, Recall, F1)

앞 지표에서는 모델의 성능을 평가할 수 있었지만, 예측 계산에대한 오차값에 근거한 값으로 직관적인 해석이 어려웠습니다. 여기에서는 실제 샘플을 몇개 맞추었는가 등과 같은 데이터 샘플 자체에 채점을 하는 방식의 지표를 통해 직관적인 모델 해석을 할 수 있도록 합니다.
    
심장병 여부는 <b>이진 분류(0 또는 1)</b> 문제이므로, 예측값을 **0~1 범위로 클리핑(clipping)** 한 뒤, 특정 기준값(threshold, 기본 0.5)을 기준으로 0/1로 변환합니다.  

그 후, 실제 라벨과 비교하여 아래 네 가지 대표적인 분류 지표를 계산합니다.

- **Accuracy (정확도)**: 전체 샘플 중 맞게 예측한 비율  
- **Precision (정밀도)**: 예측을 양성(심장병 있음)이라고 한 것 중 실제로 맞은 비율  
- **Recall (재현율)**: 실제 양성(심장병 있음) 중에서 놓치지 않고 맞춘 비율  
- **F1 Score**: 정밀도와 재현율의 조화 평균  

이 과정을 통해 KNN 방식, Median/Mode 방식 두 가지 전처리 버전에서 학습된 선형 회귀 모델의 분류 성능을 간단히 비교할 수 있습니다.


In [ ]:
# ------------------------------------------------------------
# 숫자만 출력 (ACC / PREC / RECALL / F1)  — ROC-AUC, CM 전부 제거
# ------------------------------------------------------------
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

THRESH = 0.5  # 필요시 조정

# 회귀 예측값을 0~1로 클리핑
scores_knn = np.clip(y_pred_knn_linear, 0.0, 1.0)
scores_mm  = np.clip(y_pred_mm_linear,  0.0, 1.0)

def eval_min(name, y_true, scores, thr=0.5):
    y_true = np.asarray(y_true)
    y_hat  = (scores >= thr).astype(int)

    acc  = accuracy_score(y_true, y_hat)
    prec = precision_score(y_true, y_hat, zero_division=0)
    rec  = recall_score(y_true, y_hat, zero_division=0)
    f1   = f1_score(y_true, y_hat, zero_division=0)

    print(f"{name} -> ACC={acc:.3f}, PREC={prec:.3f}, RECALL={rec:.3f}, F1={f1:.3f}")

eval_min("Linear (KNN)", y_test_knn, scores_knn, THRESH)
eval_min("Linear (MM) ",  y_test_mm,  scores_mm,  THRESH)


#### <b> 예측 결과 해석

- ACC는 약 80%가 나오지만 Recall과 Precision이 모두 0이고 따라서 F1스코어도 0이 나오게 됩니다. 사실상 예측값이 다수인 클래스로 예측된것이라 볼 수 있으며, 선형 회귀모델이 이 데이터에 적합하지 않은것으로 보입니다. 다음 차수에 선형회귀가 아닌 다른 모델을 통해 동일한 데이터세트를 예측해보도록 하겠습니다.
----


## 7. 임의의 환자 심장병 유무 예측하기

학습된 모델이 실제로 어떻게 작동하는지 확인하기 위해, <b>테스트 데이터에서 임의의 환자 1명</b>을 선택하여 예측을 수행합니다. 단순한 지표(MSE, R² 등)로만 모델 성능을 평가하는 것을 넘어, 실제 개별 환자 수준에서 모델이 어떻게 의사결정을 내리는지를 보여주는 사례가 됩니다. 이를 통해 모델의 <b>실질적 활용 가능성</b>을 직관적으로 이해할 수 있습니다.    

선택된 환자의 여러 건강 지표(나이, 혈압, 콜레스테롤 수치, 생활습관 등)를 입력으로 넣으면, 모델은 이를 바탕으로 해당 환자가 <b>심장병이 있을지(1)</b> 혹은 <b>없을지(0)</b>를 예측합니다.  

이 코드를 반복해서 수행하면 매번 선택되는 환자가 다릅니다. 여러번 수행하며 예측결과를 확인해보세요.


In [ ]:
# 1) 두 전처리 버전의 테스트 인덱스 교집합에서 1명을 무작위 선택
common_idx = list(set(X_test_knn.index).intersection(set(X_test_mm.index)))
assert len(common_idx) > 0, "교집합 인덱스가 비었습니다. 앞 단계 분할 코드를 확인하세요."
sel_idx = random.choice(common_idx)

# 2) 원본 환자 피처(라벨 포함) 확인
print(f"[선택된 환자 index] {sel_idx}")
patient_raw = df.loc[sel_idx]
display(pd.DataFrame([patient_raw]).reset_index(drop=True))

# 3) 전처리된 입력 행 준비 (모델에 그대로 투입)
x_knn = X_test_knn.loc[[sel_idx]]   # 1×P DataFrame
x_mm  = X_test_mm.loc[[sel_idx]]    # 1×P DataFrame
gt    = int(y_test_knn.loc[sel_idx])  # 정답 라벨(0/1)

# 4) 두 모델 예측 (LinearRegression → 실수 스코어, 0~1로 클리핑 후 0.5 임계값)
yhat_knn = float(model_knn_linear.predict(x_knn)[0])
yhat_mm  = float(model_mm_linear.predict(x_mm)[0])

score_knn = float(np.clip(yhat_knn, 0.0, 1.0))
score_mm  = float(np.clip(yhat_mm,  0.0, 1.0))

pred_knn = int(score_knn >= 0.5)
pred_mm  = int(score_mm  >= 0.5)

# 5) 요약표 출력
summary = pd.DataFrame([
    {"Model": "Linear (KNN)", "Score(≈risk)": round(score_knn, 3), "Pred": pred_knn, "GT": gt},
    {"Model": "Linear (MM)",  "Score(≈risk)": round(score_mm,  3), "Pred": pred_mm,  "GT": gt},
])
display(summary)

# 6) 해석 가이드(간단)
print(
    "해석) Score는 0~1 사이의 연속값(선형회귀 출력값을 0~1로 클리핑)으로, "
    "0.5 이상이면 1로 분류하고 그 미만이면 0으로 분류합니다. "
    "Pred와 GT가 같으면 해당 모델이 올바르게 예측한 것입니다."
)
